In [6]:
import re, json
import pandas as pd
from joblib import load
import glob, os
import sys

In [13]:
from utils.exploration_utils import (
    read_csv_files,
    preprocess_data,
    generate_target_variable,
    engineer_features
)

In [7]:
# Ruta absoluta al directorio src
src_path = os.path.abspath("../src") # Adjust if the notebook location changes relative to src
if src_path not in sys.path:
    sys.path.append(src_path)

In [33]:

# 1) Extraer todos los CSV que comienzan con '20250423' y concatenarlos
rt_path      = "../data_realtime_aggregated/EURUSD/5mins/"
csv_pattern  = "20250424*.csv"
csv_files    = glob.glob(os.path.join(rt_path, csv_pattern))
df_rt_list   = []

for fp in csv_files:
    df = pd.read_csv(fp)
                     
    # Localizar a UTC
    df_rt_list.append(df)

# Concatenar y renombrar índice
df_rt_20250424 = pd.concat(df_rt_list).sort_values(by='datetime_utc').reset_index(drop=True)

# Ahora df_rt_20250423 contiene todas las velas 5m de 2025‑04‑23
print(df_rt_20250424.head())


                datetime_utc     open     high      low    close  volume  \
0  2025-04-24 13:25:00+00:00  1.13791  1.13793  1.13728  1.13743   -60.0   
1  2025-04-24 13:30:00+00:00  1.13743  1.13794  1.13729  1.13793   -60.0   
2  2025-04-24 13:35:00+00:00  1.13793  1.13798  1.13763  1.13786   -60.0   
3  2025-04-24 13:40:00+00:00  1.13786  1.13786  1.13698  1.13733   -60.0   
4  2025-04-24 13:45:00+00:00  1.13733  1.13755  1.13636  1.13647   -60.0   

   barCount  
0       -60  
1       -60  
2       -60  
3       -60  
4       -60  


In [12]:

# Asegurarse que el índice es DatetimeIndex
if not isinstance(df_rt_20250424.index, pd.DatetimeIndex):
    print("Error: El índice no es DatetimeIndex. Convirtiendo...")
    try:
        # Intentar convertir la columna índice a datetime si no lo es
        df_rt_20250424.index = pd.to_datetime(df_rt_20250424.index)
        # Asegurarse que sea UTC si no tiene timezone
        if df_rt_20250424.index.tz is None:
            df_rt_20250424.index = df_rt_20250424.index.tz_localize('UTC')
        else:
            df_rt_20250424.index = df_rt_20250424.index.tz_convert('UTC')
        print("Índice convertido a DatetimeIndex UTC.")
    except Exception as e:
        print(f"Error al convertir el índice: {e}. No se puede continuar con el chequeo.")
        # Salir o manejar el error como prefieras
        # exit()

# Número esperado de barras de 5 minutos por hora
expected_bars_per_hour = 12

# Agrupar por fecha y hora
hourly_counts = df_rt_20250424.groupby(pd.Grouper(freq='H')).size()

# Filtrar horas con menos barras de las esperadas
missing_bars_hours = hourly_counts[hourly_counts < expected_bars_per_hour]

if missing_bars_hours.empty:
    print("Chequeo OK: Todas las horas tienen las 12 barras de 5 minutos esperadas.")
else:
    print("\n¡Advertencia! Se encontraron horas con menos de 12 barras:")
    for hour, count in missing_bars_hours.items():
        print(f"  - Hora: {hour.strftime('%Y-%m-%d %H:00')} UTC, Barras encontradas: {count} (Faltan: {expected_bars_per_hour - count})")



Error: El índice no es DatetimeIndex. Convirtiendo...
Índice convertido a DatetimeIndex UTC.

¡Advertencia! Se encontraron horas con menos de 12 barras:
  - Hora: 2025-04-24 13:00 UTC, Barras encontradas: 7 (Faltan: 5)
  - Hora: 2025-04-24 21:00 UTC, Barras encontradas: 9 (Faltan: 3)
  - Hora: 2025-04-25 00:00 UTC, Barras encontradas: 1 (Faltan: 11)


In [18]:
print(os.getcwd())

c:\Users\Administrador\Documents\Proyectos\Trading\ForexDataExtractorProject\notebooks


In [25]:
# Read Data from CSV files:
folder_path = '../data/EURUSD/5mins/202504.csv' # Define the folder containing CSV files
print(f"Attempting to load data from CSV files in: {folder_path}")

df_EURUSD_5min = df = pd.read_csv(folder_path) # Use the utility function to read CSVs
df_EURUSD_5min = preprocess_data(df_EURUSD_5min)

Attempting to load data from CSV files in: ../data/EURUSD/5mins/202504.csv
Starting preprocessing...
Convirtiendo fechas/horas parseadas a UTC...
Extrayendo componentes de fecha/hora desde UTC...
Ordenando DataFrame por fecha/hora UTC...
Preprocesamiento completado. DataFrame ordenado por fecha/hora UTC.


In [35]:
# --- Añadir una nueva celda de código ---

print("\n--- Comparando df_rt_20250424 vs df_EURUSD_5min filtrado ---")

# Verificar que los DataFrames existen y no están vacíos
if 'df_rt_20250424' in locals() and not df_rt_20250424.empty and \
   'df_EURUSD_5min' in locals() and not df_EURUSD_5min.empty:

    # --- 1. Encontrar fecha mayor y menor en df_rt_20250424 ---
    try:
        # Asegurarse que la columna datetime_utc es datetime
        if not pd.api.types.is_datetime64_any_dtype(df_rt_20250424['datetime_utc']):
            print("Convirtiendo 'datetime_utc' de df_rt_20250424 a datetime...")
            df_rt_20250424['datetime_utc'] = pd.to_datetime(df_rt_20250424['datetime_utc'], utc=True)

        min_date_rt = df_rt_20250424['datetime_utc'].min()
        max_date_rt = df_rt_20250424['datetime_utc'].max()
        print(f"Rango de fechas en df_rt_20250424: {min_date_rt} a {max_date_rt}")

    except KeyError:
        print("Error: La columna 'datetime_utc' no existe en df_rt_20250424.")
        min_date_rt, max_date_rt = None, None
    except Exception as e:
        print(f"Error al obtener fechas min/max de df_rt_20250424: {e}")
        min_date_rt, max_date_rt = None, None

    # --- 2. Usar esas fechas para filtrar df_EURUSD_5min ---
    df_hist_filtered = pd.DataFrame() # Inicializar vacío
    if min_date_rt is not None and max_date_rt is not None:
        try:
            # Asegurarse que la columna 'datetime' en df_EURUSD_5min es datetime y UTC
            if 'datetime' not in df_EURUSD_5min.columns:
                 raise KeyError("La columna 'datetime' no existe en df_EURUSD_5min.")
            if not pd.api.types.is_datetime64_any_dtype(df_EURUSD_5min['datetime']):
                print("Convirtiendo 'datetime' de df_EURUSD_5min a datetime...")
                df_EURUSD_5min['datetime'] = pd.to_datetime(df_EURUSD_5min['datetime'], utc=True)
            # Asegurar UTC
            if df_EURUSD_5min['datetime'].dt.tz is None:
                 df_EURUSD_5min['datetime'] = df_EURUSD_5min['datetime'].dt.tz_localize('UTC')
            else:
                 df_EURUSD_5min['datetime'] = df_EURUSD_5min['datetime'].dt.tz_convert('UTC')


            # Filtrar usando el rango de fechas obtenido
            df_hist_filtered = df_EURUSD_5min[
                (df_EURUSD_5min['datetime'] >= min_date_rt) &
                (df_EURUSD_5min['datetime'] <= max_date_rt)
            ].copy()

            if df_hist_filtered.empty:
                print("Advertencia: No se encontraron datos en df_EURUSD_5min para el rango de fechas especificado.")
            else:
                print(f"df_EURUSD_5min filtrado entre {min_date_rt} y {max_date_rt}. {len(df_hist_filtered)} filas encontradas.")
                print(df_hist_filtered.head())

        except KeyError as e:
            print(f"Error: {e}")
        except Exception as e:
            print(f"Error al filtrar df_EURUSD_5min: {e}")

    # --- 3. Comparar ambos DataFrames ---
    if not df_rt_20250424.empty and not df_hist_filtered.empty:

        # a) Comparación de cantidad de registros por hora
        print("\n--- Comparando conteo de barras por hora ---")
        hourly_counts_rt = df_rt_20250424.groupby(df_rt_20250424['datetime_utc'].dt.floor('H')).size().rename('count_rt')
        hourly_counts_hist = df_hist_filtered.groupby(df_hist_filtered['datetime'].dt.floor('H')).size().rename('count_hist')

        hourly_comparison = pd.concat([hourly_counts_rt, hourly_counts_hist], axis=1).fillna(0).astype(int)
        hourly_comparison['diff'] = hourly_comparison['count_rt'] - hourly_comparison['count_hist']

        print(hourly_comparison)
        if (hourly_comparison['diff'] != 0).any():
            print("\n¡Advertencia! Se encontraron diferencias en el número de barras por hora:")
            print(hourly_comparison[hourly_comparison['diff'] != 0])
        else:
            print("\nChequeo OK: El número de barras por hora coincide entre ambos DataFrames.")

        # b) Comparación vela a vela (Merge)
        print("\n--- Comparando datos vela a vela ---")
        # Renombrar columnas OHLCV para evitar conflictos en el merge
        cols_to_compare = ['open', 'high', 'low', 'close', 'volume']
        # Asegurarse que las columnas existen en ambos dataframes antes de incluirlas
        valid_cols_rt = [col for col in cols_to_compare if col in df_rt_20250424.columns]
        valid_cols_hist = [col for col in cols_to_compare if col in df_hist_filtered.columns]
        # Usar la intersección de columnas válidas para la comparación
        final_cols_to_compare = list(set(valid_cols_rt) & set(valid_cols_hist))
        if not final_cols_to_compare:
             print("Error: No hay columnas comunes (OHLCV) para comparar entre los DataFrames.")
        else:
            print(f"Columnas a comparar: {final_cols_to_compare}")
            df_rt_renamed = df_rt_20250424[['datetime_utc'] + final_cols_to_compare].rename(columns={'datetime_utc': 'datetime'})
            df_hist_renamed = df_hist_filtered[['datetime'] + final_cols_to_compare]

            # Realizar merge outer para encontrar diferencias y faltantes
            df_merged = pd.merge(
                df_rt_renamed,
                df_hist_renamed,
                on='datetime',
                how='outer',
                suffixes=('_rt', '_hist'),
                indicator=True # Añade columna _merge ('left_only', 'right_only', 'both')
            )

            # Filas presentes solo en Realtime (df_rt_20250424)
            missing_in_hist = df_merged[df_merged['_merge'] == 'left_only']
            if not missing_in_hist.empty:
                print(f"\n{len(missing_in_hist)} Velas encontradas en Realtime pero NO en Histórico:")
                print(missing_in_hist[['datetime']])

            # Filas presentes solo en Histórico (df_hist_filtered)
            missing_in_rt = df_merged[df_merged['_merge'] == 'right_only']
            if not missing_in_rt.empty:
                print(f"\n{len(missing_in_rt)} Velas encontradas en Histórico pero NO en Realtime:")
                print(missing_in_rt[['datetime']])

            # Filas presentes en ambos: Calcular diferencias en OHLCV
            df_both = df_merged[df_merged['_merge'] == 'both'].copy()
            if not df_both.empty:
                total_common_rows = len(df_both)
                print(f"\n{total_common_rows} Velas encontradas en AMBOS DataFrames. Calculando diferencias...")
                diff_found_overall = False
                diff_counts = {} # Diccionario para guardar conteos de diferencias por columna

                for col in final_cols_to_compare:
                    col_rt = f'{col}_rt'
                    col_hist = f'{col}_hist'
                    # Convertir a numérico por si acaso, errores a NaN
                    df_both[col_rt] = pd.to_numeric(df_both[col_rt], errors='coerce')
                    df_both[col_hist] = pd.to_numeric(df_both[col_hist], errors='coerce')
                    # Calcular diferencia absoluta y guardarla
                    df_both[f'{col}_diff'] = (df_both[col_rt] - df_both[col_hist]).abs()

                    # Contar cuántas filas tienen una diferencia significativa
                    diff_mask = df_both[f'{col}_diff'] > 1e-9 # Usar umbral pequeño
                    count = diff_mask.sum()
                    diff_counts[col] = count

                    if count > 0:
                        diff_found_overall = True
                        # print(f"  - Se encontraron {count} diferencias en la columna '{col}'.") # Mensaje opcional dentro del bucle

                if not diff_found_overall:
                    print("  - Chequeo OK: No se encontraron diferencias significativas en los valores OHLCV para las velas comunes.")
                else:
                    print("\n  Resumen de diferencias encontradas (vela a vela):")
                    for col, count in diff_counts.items():
                         if count > 0:
                              print(f"    - Columna '{col}': {count} filas con diferencias (de {total_common_rows} filas comunes)")
                         # else: # Opcional: mostrar columnas sin diferencias
                         #     print(f"    - Columna '{col}': Sin diferencias significativas.")

                    # Mostrar resumen estadístico de las diferencias absolutas
                    diff_cols = [f'{col}_diff' for col in final_cols_to_compare]
                    print("\n  Resumen estadístico de las diferencias absolutas (para velas comunes):")
                    print(df_both[diff_cols].describe())

            else:
                print("\nNo se encontraron velas comunes entre los DataFrames para comparar valores.")

    elif df_rt_20250424.empty:
         print("No se puede comparar porque df_rt_20250424 está vacío.")
    elif df_hist_filtered.empty:
         print("No se puede comparar porque df_hist_filtered está vacío (no se encontraron datos históricos en el rango).")

else:
    print("Error: No se encontraron los DataFrames 'df_rt_20250424' o 'df_EURUSD_5min' o están vacíos.")



--- Comparando df_rt_20250424 vs df_EURUSD_5min filtrado ---
Rango de fechas en df_rt_20250424: 2025-04-24 13:25:00+00:00 a 2025-04-25 00:00:00+00:00
df_EURUSD_5min filtrado entre 2025-04-24 13:25:00+00:00 y 2025-04-25 00:00:00+00:00. 125 filas encontradas.
                                         open      high       low     close  \
1970-01-01 00:00:00.000004925+00:00  1.137910  1.137935  1.137275  1.137435   
1970-01-01 00:00:00.000004926+00:00  1.137435  1.137935  1.137285  1.137935   
1970-01-01 00:00:00.000004927+00:00  1.137935  1.137980  1.137625  1.137860   
1970-01-01 00:00:00.000004928+00:00  1.137860  1.137860  1.136980  1.137330   
1970-01-01 00:00:00.000004929+00:00  1.137330  1.137545  1.136350  1.136465   

                                     volume  barCount  \
1970-01-01 00:00:00.000004925+00:00      -1        -1   
1970-01-01 00:00:00.000004926+00:00      -1        -1   
1970-01-01 00:00:00.000004927+00:00      -1        -1   
1970-01-01 00:00:00.000004928+00:00   

In [36]:
# --- Añadir NUEVA celda de código DESPUÉS de la celda con id: "df13899f" ---

print("\n--- Preparando Historial Extendido para Ingeniería de Features ---")

# Número de velas históricas adicionales a incluir antes del inicio del día
N_HIST_CANDLES_BUFFER = 150
INTERVAL_MINUTES = 5 # Asumiendo velas de 5 minutos

# Verificar que las variables necesarias existen
if 'df_EURUSD_5min' in locals() and not df_EURUSD_5min.empty and \
   'min_date_rt' in locals() and min_date_rt is not None and \
   'max_date_rt' in locals() and max_date_rt is not None:

    try:
        # Calcular la fecha de inicio necesaria para el historial
        hist_start_needed = min_date_rt - pd.Timedelta(minutes=N_HIST_CANDLES_BUFFER * INTERVAL_MINUTES)
        print(f"Fecha mínima de datos realtime (min_date_rt): {min_date_rt}")
        print(f"Fecha máxima de datos realtime (max_date_rt): {max_date_rt}")
        print(f"Fecha de inicio calculada para historial extendido: {hist_start_needed}")

        # Asegurar que la columna 'datetime' en df_EURUSD_5min es datetime UTC (ya debería estarlo por celdas anteriores)
        if not pd.api.types.is_datetime64_any_dtype(df_EURUSD_5min['datetime']) or df_EURUSD_5min['datetime'].dt.tz is None:
             print("Advertencia: La columna 'datetime' de df_EURUSD_5min no está en formato Datetime UTC. Intentando convertir...")
             df_EURUSD_5min['datetime'] = pd.to_datetime(df_EURUSD_5min['datetime'], utc=True)
             print("Columna 'datetime' convertida a UTC.")

        # Filtrar df_EURUSD_5min para obtener el historial extendido
        df_hist_for_eng = df_EURUSD_5min[
            (df_EURUSD_5min['datetime'] >= hist_start_needed) &
            (df_EURUSD_5min['datetime'] <= max_date_rt)
        ].copy()

        # Verificar si se encontraron datos
        if df_hist_for_eng.empty:
            print(f"Advertencia: No se encontraron datos en df_EURUSD_5min entre {hist_start_needed} y {max_date_rt}.")
        else:
            # Establecer 'datetime' como índice (útil para engineer_features)
            df_hist_for_eng.set_index('datetime', inplace=True)
            df_hist_for_eng.sort_index(inplace=True)
            print(f"\nHistorial extendido 'df_hist_for_eng' creado con {len(df_hist_for_eng)} filas.")
            print(f"Rango de fechas en df_hist_for_eng: {df_hist_for_eng.index.min()} a {df_hist_for_eng.index.max()}")
            print("Primeras filas de df_hist_for_eng:")
            print(df_hist_for_eng.head())
            print("\nÚltimas filas de df_hist_for_eng:")
            print(df_hist_for_eng.tail())

    except Exception as e:
        print(f"Error al crear el historial extendido: {e}")
        df_hist_for_eng = pd.DataFrame() # Asegurar que esté vacío en caso de error

else:
    print("Error: No se puede crear el historial extendido. Falta 'df_EURUSD_5min' o las fechas 'min_date_rt'/'max_date_rt' no están definidas (asegúrate de ejecutar la celda anterior).")
    df_hist_for_eng = pd.DataFrame() # Definir como vacío



--- Preparando Historial Extendido para Ingeniería de Features ---
Fecha mínima de datos realtime (min_date_rt): 2025-04-24 13:25:00+00:00
Fecha máxima de datos realtime (max_date_rt): 2025-04-25 00:00:00+00:00
Fecha de inicio calculada para historial extendido: 2025-04-24 00:55:00+00:00

Historial extendido 'df_hist_for_eng' creado con 275 filas.
Rango de fechas en df_hist_for_eng: 2025-04-24 00:55:00+00:00 a 2025-04-25 00:00:00+00:00
Primeras filas de df_hist_for_eng:
                               open      high       low     close  volume  \
datetime                                                                    
2025-04-24 00:55:00+00:00  1.133970  1.134695  1.133970  1.134380      -1   
2025-04-24 01:00:00+00:00  1.134380  1.134920  1.134325  1.134505      -1   
2025-04-24 01:05:00+00:00  1.134505  1.134635  1.134015  1.134235      -1   
2025-04-24 01:10:00+00:00  1.134235  1.134760  1.134020  1.134515      -1   
2025-04-24 01:15:00+00:00  1.134515  1.135225  1.134215  1.134

In [38]:
# --- Añadir NUEVA celda de código ANTES de la celda con id: "8944fbf7" ---

import joblib
import json
import os
import traceback

print("--- Cargando Modelo, Clusterer y Metadatos ---")

# --- Definir Rutas (Asegúrate que coincidan con las usadas al guardar) ---
model_dir = '../models/EURUSD'
model_filename = 'forex_rf_pipeline_eurusd_5mins.joblib'
metadata_filename = 'forex_rf_pipeline_eurusd_5mins_metadata.json'
clusterer_filename = 'forex_candle_clusterer_eurusd_5mins.joblib'

model_filepath = os.path.join(model_dir, model_filename)
metadata_filepath = os.path.join(model_dir, metadata_filename)
clusterer_filepath = os.path.join(model_dir, clusterer_filename)

# --- Inicializar variables ---
simplified_pipeline = None
fitted_clusterer = None
metadata = None
N_CANDLES_PARAM = None
N_CLUSTERS_PARAM = None
MIN_PATTERN_THRESHOLD_PARAM = 100 # Valor por defecto, se intentará sobreescribir desde metadata si existe

# --- Cargar Artefactos ---
try:
    # Cargar Pipeline
    if os.path.exists(model_filepath):
        simplified_pipeline = joblib.load(model_filepath)
        print(f"  - Pipeline cargado desde: {model_filepath}")
    else:
        raise FileNotFoundError(f"Archivo de pipeline no encontrado: {model_filepath}")

    # Cargar Clusterer
    if os.path.exists(clusterer_filepath):
        fitted_clusterer = joblib.load(clusterer_filepath)
        print(f"  - Clusterer cargado desde: {clusterer_filepath}")
    else:
        raise FileNotFoundError(f"Archivo de clusterer no encontrado: {clusterer_filepath}")

    # Cargar Metadata
    if os.path.exists(metadata_filepath):
        with open(metadata_filepath, 'r') as f:
            metadata = json.load(f)
        print(f"  - Metadata cargado desde: {metadata_filepath}")

        # Extraer parámetros desde metadata
        N_CANDLES_PARAM = metadata.get('n_candles_param')
        N_CLUSTERS_PARAM = metadata.get('n_clusters_param')
        # Intentar cargar MIN_PATTERN_THRESHOLD_PARAM si se guardó, si no, mantener el default
        MIN_PATTERN_THRESHOLD_PARAM = metadata.get('min_pattern_threshold', MIN_PATTERN_THRESHOLD_PARAM)

        if N_CANDLES_PARAM is None or N_CLUSTERS_PARAM is None:
             print("  - Advertencia: 'n_candles_param' o 'n_clusters_param' no encontrados en metadata.")
        else:
             print(f"  - Parámetros extraídos: N_CANDLES={N_CANDLES_PARAM}, N_CLUSTERS={N_CLUSTERS_PARAM}, MIN_PATTERN_THRESHOLD={MIN_PATTERN_THRESHOLD_PARAM}")

    else:
        raise FileNotFoundError(f"Archivo de metadata no encontrado: {metadata_filepath}")

    print("\n--- Carga completada ---")

except FileNotFoundError as fnf_error:
    print(f"\nERROR al cargar: {fnf_error}")
    traceback.print_exc()
    # Resetear variables en caso de error
    simplified_pipeline = None
    fitted_clusterer = None
    metadata = None
    N_CANDLES_PARAM = None
    N_CLUSTERS_PARAM = None
except Exception as e:
    print(f"\nERROR inesperado durante la carga: {e}")
    traceback.print_exc()
    # Resetear variables en caso de error
    simplified_pipeline = None
    fitted_clusterer = None
    metadata = None
    N_CANDLES_PARAM = None
    N_CLUSTERS_PARAM = None


--- Cargando Modelo, Clusterer y Metadatos ---
  - Pipeline cargado desde: ../models/EURUSD\forex_rf_pipeline_eurusd_5mins.joblib
  - Clusterer cargado desde: ../models/EURUSD\forex_candle_clusterer_eurusd_5mins.joblib
  - Metadata cargado desde: ../models/EURUSD\forex_rf_pipeline_eurusd_5mins_metadata.json
  - Parámetros extraídos: N_CANDLES=15, N_CLUSTERS=5, MIN_PATTERN_THRESHOLD=100

--- Carga completada ---


In [ ]:
# --- Añadir NUEVA celda de código DESPUÉS de la celda con id: "8944fbf7" ---

import traceback
import numpy as np # Para manejar NaN si es necesario

print("\n--- Aplicando Pipeline y Predicción al Historial Extendido (df_hist_for_eng) ---")

# Inicializar DataFrame de resultados
df_hist_predictions = pd.DataFrame()

try:
    # --- 1. Verificar Prerrequisitos ---
    required_vars_pred = [
        'df_hist_for_eng', 'simplified_pipeline', 'fitted_clusterer', 'metadata',
        'N_CANDLES_PARAM', 'N_CLUSTERS_PARAM', 'MIN_PATTERN_THRESHOLD_PARAM', # Asegúrate que MIN_PATTERN_THRESHOLD_PARAM esté definido
        'min_date_rt', 'max_date_rt' # Para filtrar al final
    ]
    # Añadir MIN_PATTERN_THRESHOLD_PARAM si no existe (ajusta el valor si es diferente)
    if 'MIN_PATTERN_THRESHOLD_PARAM' not in locals():
        print("Advertencia: MIN_PATTERN_THRESHOLD_PARAM no definido. Usando valor por defecto de 100.")
        MIN_PATTERN_THRESHOLD_PARAM = 100

    missing_vars = [var for var in required_vars_pred if var not in locals() or locals()[var] is None]
    if missing_vars:
        raise NameError(f"Faltan variables necesarias para la predicción: {missing_vars}. Asegúrate de ejecutar las celdas anteriores (carga/entrenamiento de modelo, creación de df_hist_for_eng).")

    if df_hist_for_eng.empty:
        raise ValueError("El DataFrame 'df_hist_for_eng' está vacío. No se puede continuar.")

    # --- 2. Añadir Componentes de Tiempo (como en realtime_processor) ---
    print("   Añadiendo componentes de tiempo a df_hist_for_eng...")
    df_hist_for_eng_copy = df_hist_for_eng.copy() # Trabajar sobre una copia
    if pd.api.types.is_datetime64_any_dtype(df_hist_for_eng_copy.index) and df_hist_for_eng_copy.index.tz == pytz.UTC:
        df_hist_for_eng_copy['Year'] = df_hist_for_eng_copy.index.year
        df_hist_for_eng_copy['Month'] = df_hist_for_eng_copy.index.strftime('%Y%m')
        df_hist_for_eng_copy['Day'] = df_hist_for_eng_copy.index.day
        df_hist_for_eng_copy['Hour'] = df_hist_for_eng_copy.index.hour
        df_hist_for_eng_copy['Minute'] = df_hist_for_eng_copy.index.minute
        df_hist_for_eng_copy['DayOfWeek'] = df_hist_for_eng_copy.index.dayofweek
        print("   Componentes de tiempo añadidos.")
    else:
        print("   Advertencia: El índice de df_hist_for_eng no es datetime UTC. Saltando adición de componentes de tiempo.")

    # --- 3. Aplicar Ingeniería de Características ---
    print(f"   Aplicando 'engineer_features' a {len(df_hist_for_eng_copy)} filas...")
    X_engineered_hist, _ = engineer_features(
        df_hist_for_eng_copy,
        n_candles=N_CANDLES_PARAM,
        n_clusters=N_CLUSTERS_PARAM,
        min_pattern_threshold=MIN_PATTERN_THRESHOLD_PARAM,
        clusterer=fitted_clusterer, # Usar el clusterer ajustado
        fit_clusterer=False       # NO reajustar
    )
    print(f"   Shape después de engineer_features: {X_engineered_hist.shape}")

    # --- 4. Predecir Probabilidades ---
    print("   Prediciendo probabilidades con 'simplified_pipeline'...")
    # Asegurarse que las columnas esperadas por el pipeline están presentes
    expected_features = simplified_pipeline.feature_names_in_
    missing_pred_cols = [col for col in expected_features if col not in X_engineered_hist.columns]
    if missing_pred_cols:
        raise ValueError(f"Faltan columnas {missing_pred_cols} en X_engineered_hist requeridas por el pipeline.")

    # Seleccionar y reordenar columnas para la predicción
    X_input_hist = X_engineered_hist[expected_features]

    probs_hist = simplified_pipeline.predict_proba(X_input_hist)
    class_labels = simplified_pipeline.classes_
    print(f"   Probabilidades predichas. Clases: {class_labels}")

    # Crear DataFrame de probabilidades
    probs_df = pd.DataFrame(probs_hist, index=X_input_hist.index, columns=[f"prob_{c}" for c in class_labels])

    # --- 5. Aplicar Umbrales ---
    print("   Aplicando umbrales óptimos desde metadata...")
    buy_threshold = metadata.get('optimal_buy_threshold')
    sell_threshold = metadata.get('optimal_sell_threshold')
    buy_class_name = metadata.get('buy_class_name', 'objetivo_compra_alcanzado') # Usar nombre de metadata o default
    sell_class_name = metadata.get('sell_class_name', 'objetivo_venta_alcanzado')
    neutral_class_name = metadata.get('neutral_class_name', 'no_objetivo_alcanzado')

    if buy_threshold is None or sell_threshold is None:
        raise ValueError("Umbrales óptimos ('optimal_buy_threshold', 'optimal_sell_threshold') no encontrados en metadata.")
    print(f"   Umbrales: Compra >= {buy_threshold:.4f}, Venta >= {sell_threshold:.4f}")

    # Obtener columnas de probabilidad correspondientes
    prob_buy_col = f"prob_{buy_class_name}"
    prob_sell_col = f"prob_{sell_class_name}"
    if prob_buy_col not in probs_df.columns or prob_sell_col not in probs_df.columns:
        raise KeyError(f"Columnas de probabilidad '{prob_buy_col}' o '{prob_sell_col}' no encontradas en probs_df.")

    # Aplicar lógica de umbrales vectorizada
    conditions = [
        (probs_df[prob_buy_col] >= buy_threshold) & (probs_df[prob_buy_col] >= probs_df[prob_sell_col]),
        (probs_df[prob_sell_col] >= sell_threshold) & (probs_df[prob_sell_col] > probs_df[prob_buy_col])
    ]
    choices = [buy_class_name, sell_class_name]
    probs_df['action_hist'] = np.select(conditions, choices, default=neutral_class_name)

    # --- 6. Combinar Resultados y Filtrar ---
    print("   Combinando resultados y filtrando por rango de fechas original...")
    # Unir probabilidades y acción con los datos OHLCV originales del historial ingenierizado
    df_hist_predictions = df_hist_for_eng[['open', 'high', 'low', 'close', 'volume']].join(probs_df, how='inner')

    # Filtrar para mantener solo las predicciones dentro del rango de df_rt_20250424
    df_hist_predictions = df_hist_predictions[
        (df_hist_predictions.index >= min_date_rt) &
        (df_hist_predictions.index <= max_date_rt)
    ]

    # --- 7. Mostrar Resultados ---
    if df_hist_predictions.empty:
        print("\nAdvertencia: No se generaron predicciones para el rango de fechas especificado.")
    else:
        print(f"\n--- Predicciones Históricas Recalculadas ({len(df_hist_predictions)} filas) ---")
        print(df_hist_predictions.head())
        print("\nDistribución de acciones recalculadas:")
        print(df_hist_predictions['action_hist'].value_counts())

except NameError as ne:
    print(f"\nERROR: {ne}")
    traceback.print_exc()
except ValueError as ve:
    print(f"\nERROR: {ve}")
    traceback.print_exc()
except KeyError as ke:
    print(f"\nERROR: {ke}")
    traceback.print_exc()
except Exception as e:
    print(f"\nERROR inesperado durante la predicción histórica: {e}")
    traceback.print_exc()

print("\n--- Celda de Predicción Histórica Finalizada ---")



--- Aplicando Pipeline y Predicción al Historial Extendido (df_hist_for_eng) ---

ERROR: Faltan variables necesarias para la predicción: ['df_hist_for_eng', 'simplified_pipeline', 'fitted_clusterer', 'metadata', 'N_CANDLES_PARAM', 'N_CLUSTERS_PARAM', 'MIN_PATTERN_THRESHOLD_PARAM', 'min_date_rt', 'max_date_rt']. Asegúrate de ejecutar las celdas anteriores (carga/entrenamiento de modelo, creación de df_hist_for_eng).

--- Celda de Predicción Histórica Finalizada ---


Traceback (most recent call last):
  File "C:\Users\Administrador\AppData\Local\Temp\ipykernel_46084\2575853747.py", line 25, in <module>
    raise NameError(f"Faltan variables necesarias para la predicción: {missing_vars}. Asegúrate de ejecutar las celdas anteriores (carga/entrenamiento de modelo, creación de df_hist_for_eng).")
NameError: Faltan variables necesarias para la predicción: ['df_hist_for_eng', 'simplified_pipeline', 'fitted_clusterer', 'metadata', 'N_CANDLES_PARAM', 'N_CLUSTERS_PARAM', 'MIN_PATTERN_THRESHOLD_PARAM', 'min_date_rt', 'max_date_rt']. Asegúrate de ejecutar las celdas anteriores (carga/entrenamiento de modelo, creación de df_hist_for_eng).


In [32]:
df_EURUSD_5min.datetime

1970-01-01 00:00:00+00:00             2025-04-01 04:00:00+00:00
1970-01-01 00:00:00.000000001+00:00   2025-04-01 04:05:00+00:00
1970-01-01 00:00:00.000000002+00:00   2025-04-01 04:10:00+00:00
1970-01-01 00:00:00.000000003+00:00   2025-04-01 04:15:00+00:00
1970-01-01 00:00:00.000000004+00:00   2025-04-01 04:20:00+00:00
                                                 ...           
1970-01-01 00:00:00.000005217+00:00   2025-04-25 14:00:00+00:00
1970-01-01 00:00:00.000005218+00:00   2025-04-25 14:05:00+00:00
1970-01-01 00:00:00.000005219+00:00   2025-04-25 14:10:00+00:00
1970-01-01 00:00:00.000005220+00:00   2025-04-25 14:15:00+00:00
1970-01-01 00:00:00.000005221+00:00   2025-04-25 14:20:00+00:00
Name: datetime, Length: 5222, dtype: datetime64[ns, UTC]

In [ ]:


# 3) Unir ambas fuentes por timestamp
df_bars = df_rt_agg.join(
    df_hist[["open","high","low","close"]].rename(
        columns=lambda c: c+"_hist"
    ),
    how="inner",
    lsuffix="_rt"
).sort_index()

# 4) Cargar logs de predicción realtime
pattern = re.compile(r"@ (?P<ts>[\d\- :]+).*?entry_price: (?P<price>[\d\.]+).*?Predicción Final.*?: (?P<act>\w+)", re.DOTALL)
recs = []
for line in open("data_realtime_aggregated/realtime_processor.log"):
    m = pattern.search(line)
    if m:
        ts = pd.to_datetime(m.group("ts")).tz_localize("UTC")
        recs.append(dict(ts=ts, action_rt=m.group("act"), price_rt=float(m.group("price"))))
df_logs = pd.DataFrame(recs).set_index("ts")

# 5) Cargar pipeline y umbrales
pipe = load("models/EURUSD/forex_rf_pipeline_eurusd_5mins.joblib")
meta = json.load(open("models/EURUSD/forex_rf_pipeline_eurusd_5mins_metadata.json"))
thr_b, thr_s = meta["optimal_buy_threshold"], meta["optimal_sell_threshold"]

# 6) Para cada evento, predecir con barras realtime vs barras históricas
out = []
for ts, log in df_logs.iterrows():
    if ts not in df_bars.index: continue
    row = df_bars.loc[ts]
    # crear X a partir de la fila histórica
    X = pd.DataFrame([row[["open_rt","high_rt","low_rt","close_rt",
                           "open_hist","high_hist","low_hist","close_hist"]]])
    probs = pipe.predict_proba(X)[0]
    cls = pipe.classes_.tolist()
    act_hist = (
        "buy" if probs[cls.index("objetivo_compra_alcanzado")]>=thr_b else
        "sell" if probs[cls.index("objetivo_venta_alcanzado")]>=thr_s else
        "none"
    )
    out.append({
        "ts": ts,
        "price_rt": log["price_rt"],
        "action_rt": log["action_rt"],
        "price_close_rt": row["close_rt"],
        "price_close_hist": row["close_hist"],
        "action_hist": act_hist
    })
df_cmp = pd.DataFrame(out).set_index("ts")
print(df_cmp)
df_cmp.to_csv("comparison_rt_vs_hist.csv")